In [1]:
import pandas as pd
import importlib
import OHE_chrono
from datetime import datetime
from scipy import stats

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error as mape 
from sklearn.preprocessing import StandardScaler as scaler
import numpy as np


In [22]:
# we reduce the size of the dataset, if needed, to accomodate for weaker machine
db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
db_plot = db.copy() 
non_na_rows = db[~db["System"].isna()]
na_rows = db[db["System"].isna()].sample(n=5000, random_state=42)  # Adjust 'n' as needed
db_sampled = pd.concat([non_na_rows, na_rows])
db_sampled.sort_values(by=["Date_Time", "EPISODE_ID"], inplace=True)
db=db_sampled.copy()
db = db.drop(columns= ["Date_Time"])

target = "RUL"
column_names = db.columns.tolist()
numerical_columns = [col for col in column_names if db[col].dtype == np.float64 and col != target and col not in ["EPISODE_ID"]]
categorical_columns = [col for col in column_names if db[col].dtype != np.float64 and col != target]

# Not good to dropna and cause it might be useful information
db = db.dropna()

#All the categorical columns need to be treated as boolean to use for regression, so we create a one hot encoding
## for all the categorical columns
db = pd.get_dummies(db, columns=categorical_columns, drop_first=True)

#separate de database, the first 11 Episodes are for training purposes 
x_test = db[db["EPISODE_ID"] >= 11]
x_train = db[~(db["EPISODE_ID"] >= 11)]
x_test = x_test.drop(columns="EPISODE_ID")
x_train = x_train.drop(columns="EPISODE_ID")
# Scaling of the numericals columns data to prevent any scaling mismatch between columns
x_train[numerical_columns] = scaler().fit_transform(x_train[numerical_columns])
x_test[numerical_columns] = scaler().fit_transform(x_test[numerical_columns])

# Splitting of the data between trainning set and testing set

X_train, X_test, y_train, y_test = x_train.drop(columns="RUL"), x_test.drop(columns="RUL"), x_train["RUL"], x_test["RUL"]


In [26]:
### Plotting RUL 


In [ ]:
reg = LinearRegression().fit(X_train,y_train)
y_predicted = reg.predict(X_train)

# Metrics Evaluation
R2 = reg.score(X_train,y_train)


n = X_train.shape[0]
p = X_train.shape[1]
y_mean = np.mean(y_train)
sst = np.sum((y_train - y_mean)**2)
ssr = np.sum((y_predicted - y_mean)**2)
sse = np.sum((y_train - y_predicted)**2)


msr = ssr / p
mse = sse / (n - p - 1)
f_stat = msr / mse
f_pval = stats.f.sf(f_stat, p, n - p - 1)


print(f"R^2 (sklearn):{R2}")
print(f"R^2 (computed) {(1-sse/sst)}")
print(f"F-statistic: {f_stat:.4f}")
print(f"F p-value: {f_pval:.4f}")

R^2 (sklearn):0.9702484469625023


R^2 (computed) 0.9702484469625023
F-statistic: 264.9419
F p-value: 0.0000


In [27]:
class LearningModel:
    """


    """
    def __init__(self,  reject_columns, target):
        self.target = target
        self.reject_columns = reject_columns
        ## puisse gerer different model et en donner differents
    
    def prepare_data(self, db, target):

    # we reduce the size of the dataset, if needed, to accomodate for weaker machine
        column_names = db.columns.tolist()
        db_plot = db.copy() 
        non_na_rows = db[~db["System"].isna()]
        na_rows = db[db["System"].isna()].sample(n=5000, random_state=42)
        db_sampled = pd.concat([non_na_rows, na_rows])

        # FIX: Check if Date_Time exists before sorting
        if "Date_Time" in db_sampled.columns:
            db_sampled.sort_values(by=["Date_Time", "EPISODE_ID"], inplace=True)
            db = db_sampled.copy()
            #db = db.drop(columns=["Date_Time"])  # Only drop if it exists
        else:
            db_sampled.sort_values(by=["EPISODE_ID"], inplace=True)
            db = db_sampled.copy()

        numerical_columns = [col for col in column_names if db[col].dtype == np.float64 and col != target and col not in self.reject_columns ]
        categorical_columns = [col for col in column_names if db[col].dtype != np.float64 and col != target]
        # Not good to dropna and cause it might be useful information
        db = db.dropna()

        #All the categorical columns need to be treated as boolean to use for regression, so we create a one hot encoding
        ## for all the categorical columns
        db = pd.get_dummies(db, columns=categorical_columns, drop_first=True)

        return self.split_train_test(db, numerical_columns)

    def split_train_test(self, db, numerical_columns):
        #separate de database, the first 11 Episodes are for training purposes 
        x_test = db[db["EPISODE_ID"] >= 11]
        x_train = db[~(db["EPISODE_ID"] >= 11)]
        #x_test = x_test.drop(columns="EPISODE_ID")
        #x_train = x_train.drop(columns="EPISODE_ID")
        # Scaling of the numericals columns data to prevent any scaling mismatch between columns
        x_train[numerical_columns] = scaler().fit_transform(x_train[numerical_columns])
        x_test[numerical_columns] = scaler().fit_transform(x_test[numerical_columns])
        X_train, X_test, y_train, y_test = x_train.drop(columns=self.target), x_test.drop(columns=self.target), x_train[self.target], x_test[self.target]
        data = [X_train, y_train, X_test, y_test]

        return data
        
    def linearRegression(self, data):
        reg = LinearRegression().fit(data[0],data[1])
        return reg
        
    def metrics(self, reg, data):
        X_train, y_train, X_test, y_test = data[0], data[1], data[2], data[3]
        y_predicted = reg.predict(X_train)

        # Metrics Evaluation
        R2 = reg.score(X_train,y_train)


        n = X_train.shape[0]
        p = X_train.shape[1]
        y_mean = np.mean(y_train)
        sst = np.sum((y_train - y_mean)**2)
        ssr = np.sum((y_predicted - y_mean)**2)
        sse = np.sum((y_train - y_predicted)**2)


        msr = ssr / p
        mse = sse / (n - p - 1)
        f_stat = msr / mse
        f_pval = stats.f.sf(f_stat, p, n - p - 1)


        print(f"R^2 (sklearn):{R2}")
        print(f"R^2 (computed) {(1-sse/sst)}")
        print(f"F-statistic: {f_stat:.4f}")
        print(f"F p-value: {f_pval:.4f}")

In [28]:
reject_columns = ["Date_Time"]
target = "RUL"

ts_model = LearningModel(reject_columns, target)
db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
data = ts_model.prepare_data(db, target)
reg = ts_model.linearRegression(data)

ts_model.metrics(reg, data) 


C:\Users\zakar\AppData\Local\Temp\ipykernel_10732\916213405.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_train[numerical_columns] = scaler().fit_transform(x_train[numerical_columns])
C:\Users\zakar\AppData\Local\Temp\ipykernel_10732\916213405.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_test[numerical_columns] = scaler().fit_transform(x_test[numerical_columns])


R^2 (sklearn):1.0
R^2 (computed) 1.0
F-statistic: 115737016173749450083139584.0000
F p-value: 0.0000


In [16]:
ts_model.metrics()

TypeError: LearningModel.metrics() missing 2 required positional arguments: 'reg' and 'data'